# Federated Learning for IDS

This notebook demonstrates federated learning with different aggregation strategies for intrusion detection.

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '../experiments')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import json

from model import NeuralNetwork
from dataset import load_processed_data, FederatedDataset
from client import FederatedClient, ClientManager
from server import FederatedServer
from strategies import FedAvg, FedProx, FedDP
from evaluate import ModelEvaluator

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Federated learning modules imported successfully")

## 1. Load and Distribute Data

In [ ]:
# Load data
data_dir = "../data/processed"
X_train, X_test, y_train, y_test = load_processed_data(data_dir)

print(f"Training data: {X_train.shape}")
print(f"Test data: {X_test.shape}")

In [ ]:
# Create federated dataset
num_clients = 10

fed_dataset = FederatedDataset(X_train, y_train, n_clients=num_clients,
                               partition_method="non-iid", alpha=0.1)

client_sizes = fed_dataset.get_client_sizes()
print(f"\nCreated federated dataset with {num_clients} clients")
print(f"Client dataset sizes: {client_sizes}")
print(f"\nTotal samples distributed: {sum(client_sizes.values())}")

In [ ]:
# Visualize data distribution
plt.figure(figsize=(12, 5))
client_ids = sorted(client_sizes.keys())
sizes = [client_sizes[cid] for cid in client_ids]

bars = plt.bar([str(cid) for cid in client_ids], sizes, color='#3498db', alpha=0.7, edgecolor='black')
for bar, size in zip(bars, sizes):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(size)}', ha='center', va='bottom', fontsize=9)

plt.xlabel('Client ID', fontsize=12)
plt.ylabel('Dataset Size', fontsize=12)
plt.title('Non-IID Data Distribution Across Clients', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Setup Federated Learning with FedAvg

In [ ]:
# Hyperparameters
input_dim = X_train.shape[1]
hidden_dims = [128, 64]
learning_rate = 0.01
batch_size = 32
num_rounds = 15
local_epochs = 5

print(f"Federated Learning Configuration:")
print(f"  Clients: {num_clients}")
print(f"  Rounds: {num_rounds}")
print(f"  Local epochs per client: {local_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")

In [ ]:
# Create global model
global_model = NeuralNetwork(input_dim=input_dim, hidden_dims=hidden_dims,
                             output_dim=1, learning_rate=learning_rate)

# Create strategy and server
strategy = FedAvg(num_clients)
server = FederatedServer(global_model, strategy, num_clients)

# Create clients
client_manager = ClientManager()

for client_id in range(num_clients):
    X_client, y_client = fed_dataset.get_client_data(client_id)
    
    client_model = NeuralNetwork(input_dim=input_dim, hidden_dims=hidden_dims,
                                output_dim=1, learning_rate=learning_rate)
    
    client = FederatedClient(
        client_id=client_id,
        model=client_model,
        X_train=X_client,
        y_train=y_client,
        X_test=X_test,
        y_test=y_test,
        batch_size=batch_size
    )
    client_manager.add_client(client)

print(f"Created {client_manager.get_num_clients()} federated clients")

## 3. Federated Training

In [ ]:
# Training
all_clients = client_manager.get_all_clients()
round_losses = []
round_accuracies = []

for round_num in range(num_rounds):
    round_result = server.federated_round(all_clients, fraction=1.0, 
                                          num_local_epochs=local_epochs)
    round_losses.append(round_result['loss'])
    round_accuracies.append(round_result['accuracy'])
    
    if (round_num + 1) % 5 == 0:
        print(f"Round {round_num + 1}/{num_rounds} completed")

print(f"\nFederated training completed!")

## 4. Training Convergence Analysis

In [ ]:
# Plot convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(range(1, num_rounds + 1), round_losses, marker='o', color='#e74c3c', linewidth=2)
ax1.set_xlabel('Round', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Federated Training Loss Convergence', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(range(1, num_rounds + 1), round_accuracies, marker='s', color='#27ae60', linewidth=2)
ax2.set_xlabel('Round', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Federated Training Accuracy Improvement', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTraining Statistics:")
print(f"  Initial loss: {round_losses[0]:.4f}")
print(f"  Final loss: {round_losses[-1]:.4f}")
print(f"  Initial accuracy: {round_accuracies[0]:.4f}")
print(f"  Final accuracy: {round_accuracies[-1]:.4f}")

## 5. Final Global Model Evaluation

In [ ]:
# Evaluate global model
evaluator = ModelEvaluator()
global_metrics = evaluator.evaluate_model(server.get_model(), X_test, y_test)
evaluator.print_metrics(global_metrics)

## 6. Client Performance Comparison

In [ ]:
# Evaluate all clients
client_accuracies = []
client_ids_list = []

for client in all_clients:
    metrics = client.evaluate()
    client_accuracies.append(metrics.get('accuracy', 0))
    client_ids_list.append(client.client_id)

# Plot
plt.figure(figsize=(12, 5))
colors = ['#3498db' if acc < global_metrics['accuracy'] else '#27ae60' 
          for acc in client_accuracies]
bars = plt.bar([str(cid) for cid in client_ids_list], client_accuracies, 
               color=colors, alpha=0.7, edgecolor='black')

plt.axhline(y=global_metrics['accuracy'], color='red', linestyle='--', 
           linewidth=2, label=f'Global Model Accuracy ({global_metrics["accuracy"]:.3f})')

plt.xlabel('Client ID', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Client Model Performance vs Global Model', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nClient Performance Summary:")
print(f"  Best client accuracy: {max(client_accuracies):.4f}")
print(f"  Worst client accuracy: {min(client_accuracies):.4f}")
print(f"  Average client accuracy: {np.mean(client_accuracies):.4f}")
print(f"  Global model accuracy: {global_metrics['accuracy']:.4f}")

## 7. Summary

In [ ]:
print("\n" + "="*60)
print("FEDERATED LEARNING SUMMARY (FedAvg)")
print("="*60)
print(f"Number of Clients: {num_clients}")
print(f"Training Rounds: {num_rounds}")
print(f"Local Epochs: {local_epochs}")
print(f"\nGlobal Model Performance:")
print(f"  Accuracy:  {global_metrics['accuracy']:.4f}")
print(f"  Precision: {global_metrics['precision']:.4f}")
print(f"  Recall:    {global_metrics['recall']:.4f}")
print(f"  F1-Score:  {global_metrics['f1']:.4f}")
print(f"\nClient Statistics:")
print(f"  Average accuracy: {np.mean(client_accuracies):.4f}")
print(f"  Accuracy variance: {np.var(client_accuracies):.4f}")
print("="*60)